In [1]:
import numpy as np
import pandas as pd
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, TargetDriftPreset

In [2]:
np.random.seed(42)

ref = pd.read_csv("data/reference.csv")
print(f"Reference samples: {len(ref)}")
print(ref.head())

cur = ref.copy()
# сдвигаем sepal length (x1.5 + шум) и petal width (x0.5)
cur["sepal length (cm)"] = cur["sepal length (cm)"] * 1.5 + np.random.normal(0, 0.1, len(cur))
cur["petal width (cm)"] = cur["petal width (cm)"] * 0.5
cur.to_csv("data/current.csv", index=False)
print(f"\nCurrent samples: {len(cur)}")
print(cur.head())

Reference samples: 120
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                4.6               3.6                1.0               0.2   
1                5.7               4.4                1.5               0.4   
2                6.7               3.1                4.4               1.4   
3                4.8               3.4                1.6               0.2   
4                4.4               3.2                1.3               0.2   

   target  
0       0  
1       0  
2       1  
3       0  
4       0  

Current samples: 120
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0           6.949671               3.6                1.0               0.1   
1           8.536174               4.4                1.5               0.2   
2          10.114769               3.1                4.4               0.7   
3           7.352303               3.4                1.6               0.1   
4           6

In [3]:
data_report = Report(metrics=[DataDriftPreset()])
data_report.run(reference_data=ref, current_data=cur)
data_dict = data_report.as_dict()

n_drifted = sum(
    m["result"]["drift_by_columns"][col]["drift_detected"]
    for m in data_dict["metrics"]
    if "drift_by_columns" in m["result"]
    for col in m["result"]["drift_by_columns"]
)
n_total = sum(
    1
    for m in data_dict["metrics"]
    if "drift_by_columns" in m["result"]
    for col in m["result"]["drift_by_columns"]
)
print(f"Data drift detected in {n_drifted}/{n_total} features")

Data drift detected in 2/5 features


In [4]:
target_report = Report(metrics=[TargetDriftPreset()])
target_report.run(reference_data=ref, current_data=cur)
target_dict = target_report.as_dict()

target_drifted = any(
    m["result"].get("drift_detected", False)
    for m in target_dict["metrics"]
)
print(f"Target drift detected: {target_drifted}")

Target drift detected: False


In [5]:
combined = Report(metrics=[DataDriftPreset(), TargetDriftPreset()])
combined.run(reference_data=ref, current_data=cur)
combined.save_html("drift_report.html")
print("drift_report.html saved")

drift_report.html saved
